In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from brain_image.utils import setup_logging


setup_logging()

In [9]:
import logging
import pandas as pd
import json
import yaml
from pathlib import Path


def get_single_file(dir: Path, pattern: str) -> Path | None:
    paths = list(dir.rglob(pattern))

    num_results = len(paths)
    if num_results == 0:
        return None

    if num_results > 1:
        raise ValueError(f"Expected to find one results matching pattern {pattern} in dir {dir} - Found {num_results}: {tuple(paths)}")

    path = paths[0]
    return path
        

def gather_metrics(experiment_dir: Path, selected_hparams: list[str] = []) -> pd.DataFrame:
    all_metrics = []

    for exp_dir in experiment_dir.iterdir():
        metrics_path = get_single_file(exp_dir, "*test/test_metrics.json")
        if metrics_path is None:
            logging.warning(f"Could not find any paths in dir {exp_dir} matching pattern {'*test/metrics.json'}")
            continue

        logging.info(f"Loading metrics from {metrics_path}")

        with open(metrics_path, "r") as f:
            metrics = json.load(f)

        if len(selected_hparams) > 0:
            hparams_path = get_single_file(exp_dir, "*hparams.yaml")
            if hparams_path is None:
                logging.warning(f"Could not find hparam file")
                continue
            
            with open(hparams_path) as f:
                hparams = yaml.safe_load(f)

            for hparam_key in selected_hparams:
                hparam_parts = hparam_key.split(".")
                curr_hparam = hparams
                for part in hparam_parts:
                    curr_hparam = curr_hparam[part]

                metrics[hparam_key] = curr_hparam

        all_metrics.append(metrics)

    metrics = pd.DataFrame.from_records(all_metrics)
    return metrics


ex_path = Path("experiments/encoders")
metrics = gather_metrics(ex_path, ["config.align_img_encoder", "config.eeg_encoder"])
metrics

12:06:58 | INFO     | Loading metrics from experiments/encoders/251025024740neldak-slurmarr14438732_0/version_0/test/test_metrics.json


12:06:58 | INFO     | Loading metrics from experiments/encoders/251025024828skzugn-slurmarr14438732_5/version_0/test/test_metrics.json
12:06:58 | INFO     | Loading metrics from experiments/encoders/251025024843tpfgkr-slurmarr14438732_7/version_0/test/test_metrics.json
12:06:58 | INFO     | Loading metrics from experiments/encoders/251025024758gdkpqh-slurmarr14438732_3/version_0/test/test_metrics.json
12:06:58 | INFO     | Loading metrics from experiments/encoders/251025024821rurqfo-slurmarr14438732_4/version_0/test/test_metrics.json
12:06:58 | INFO     | Loading metrics from experiments/encoders/251025024748wwktvu-slurmarr14438732_1/version_0/test/test_metrics.json
12:06:58 | INFO     | Loading metrics from experiments/encoders/251025024829tkdfup-slurmarr14438732_6/version_0/test/test_metrics.json
12:06:58 | INFO     | Loading metrics from experiments/encoders/251025024756lbnkuv-slurmarr14438732_2/version_0/test/test_metrics.json


,align/brain_acc,align/image_acc,prior/pixcorr,prior/ssim,prior/alex2,prior/alex5,prior/inceptionv3,prior/clip,prior/efficientnet,prior/swav,config.align_img_encoder,config.eeg_encoder
0,0.220,0.305,0.130713,0.303856,0.813769,0.854950,0.728920,0.787864,0.877443,0.557708,clip_vitl14,nice
1,0.395,0.470,0.163311,0.294539,0.828392,0.887161,0.761683,0.815704,0.871701,0.560445,unaligned_synclr_vitb16,atms
2,0.480,0.590,0.170545,0.317546,0.841482,0.892412,0.748869,0.805301,0.867542,0.555538,aligned_synclr_vitb16,atms
3,0.315,0.365,0.148432,0.306491,0.800603,0.874397,0.762437,0.813819,0.875780,0.560807,clip_vith14,atms
4,0.340,0.340,0.138662,0.315634,0.805477,0.869648,0.732186,0.820452,0.883386,0.573196,unaligned_synclr_vitb16,nice
5,0.295,0.305,0.163795,0.304130,0.792462,0.874749,0.744397,0.817010,0.866747,0.557266,clip_vitl14,atms
6,0.425,0.575,0.145043,0.312640,0.800704,0.869472,0.754975,0.824899,0.873878,0.553037,aligned_synclr_vitb16,nice
7,0.295,0.410,0.140830,0.317952,0.805050,0.845477,0.747940,0.804472,0.873470,0.561050,clip_vith14,nice
